In [ ]:
# ==============================================
# Reti-TransNet Review - Automated Colab Setup
# ==============================================

import os
import sys
import shutil

# --- 0. GPU KONTROLÜ ---
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Please go to Runtime > Change runtime type > T4 GPU.")

# --- 1. ZIP DOSYASINI İNDİR (GDOWN İLE - DÜZELTİLDİ) ---
# Google Drive Dosya ID'niz: 1dAVhBewofNvG2OMZb2tdNyKj0EjewVkB
file_id = '1dAVhBewofNvG2OMZb2tdNyKj0EjewVkB'
zip_name = 'Reti-TransNet_Review.zip'
target_dir = 'Reti-TransNet_Review'

# Eğer klasör zaten varsa temizle (Temiz kurulum için)
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

if not os.path.exists(zip_name):
    print(f"\n📥 Downloading Repository ({zip_name})...")
    # gdown kütüphanesi Colab'da yüklüdür ve Drive linklerini sorunsuz indirir
    !gdown {file_id} -O {zip_name}

if os.path.exists(zip_name):
    print("✅ Download complete.")
    print("📦 Extracting ZIP...")
    # Zip'i aç
    !unzip -q {zip_name} -d {target_dir}

    # --- KLASÖR YAPISI DÜZELTME ---
    # Bazen zip içinden bir klasör daha çıkar, onu kontrol edip içine girelim
    contents = os.listdir(target_dir)
    if len(contents) == 1 and os.path.isdir(os.path.join(target_dir, contents[0])):
        # Eğer iç içe klasör varsa (örn: Reti-TransNet-main) içine gir
        work_dir = os.path.join(target_dir, contents[0])
        os.chdir(work_dir)
    else:
        os.chdir(target_dir)

    print(f"📍 Working directory set to: {os.getcwd()}")
else:
    print("❌ CRITICAL ERROR: ZIP file could not be downloaded. Check the Google Drive Link permissions (Must be 'Anyone with the link').")
    sys.exit() # İndirme yoksa dur

# --- 2. INSTALL DEPENDENCIES ---
if os.path.exists('requirements.txt'):
    print("\n⚙️ Installing dependencies (Please wait)...")
    !pip install -r requirements.txt > /dev/null 2>&1 # Çıktıyı gizle
    print("✅ Dependencies installed.")
else:
    print("❌ ERROR: 'requirements.txt' not found inside the ZIP.")

# --- 3. KAGGLE API SETUP ---
if not os.path.exists('kaggle.json'):
    print("\n📂 Please upload your 'kaggle.json' API key now:")
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        print("✅ Kaggle key uploaded.")
    else:
        print("⚠️ Warning: 'kaggle.json' not found. Data download might fail.")

# --- 4. DATA PREPARATION ---
print("\n🚀 [1/3] Downloading Data...")
try:
    # Modül yolunu ekle (import hatası olmaması için)
    sys.path.append(os.getcwd())
    from download_data import download_datasets
    download_datasets()
except ImportError:
    print("❌ Error: 'download_data.py' not found or failed to run.")
except Exception as e:
    print(f"❌ An error occurred during download: {e}")

# --- 5. TRAINING ---
print("\n🔥 [2/3] Starting Training (25 Epochs)...")
print("    (Real-time progress will be shown below)")

# os.system yerine !python veya run kullanıyoruz ki çıktı görünsün
!python train.py

print("\n✅ Training Finished!")

# --- 6. EVALUATION ---
print("\n📊 [3/3] FINAL EVALUATION REPORT")
print("="*40)
!python evaluate.py
print("="*40)